In [3]:
import os
import glob
import pandas as pd

DATA_ROOT = os.path.join("data", "ami")

AUDIO_DIR = os.path.join(DATA_ROOT, "audio")
SUMMARY_DIR = os.path.join(DATA_ROOT, "summary")
TRANSCRIPT_DIR = os.path.join(DATA_ROOT, "transcript")

In [11]:
def parse_transcript(filepath):

    utterances = []

    with open(filepath, "r", encoding="utf-8") as f:

        for line in f:

            parts = line.strip().split("\t")

            if len(parts) != 4:
                print('Transcript line parts mismatch')
                continue

            speaker, start_time, text, section = parts

            utterances.append({
                "speaker": speaker,
                "start_time": float(start_time),
                "text": text,
                "section": section
            })

    utterance_df = pd.DataFrame(utterances)

    if len(utterance_df) == 0:
        return "", utterance_df

    # Build full transcript text
    transcript_text = " ".join(utterance_df["text"].tolist())

    return transcript_text, utterance_df

In [5]:
def load_transcripts(transcript_dir):

    records = []

    for split in ["train", "valid", "test"]:

        split_dir = os.path.join(transcript_dir, split)

        for filepath in glob.glob(os.path.join(split_dir, "*.txt")):

            meeting_id = os.path.basename(filepath).replace(".txt", "")

            transcript_text, utterance_df = parse_transcript(filepath)

            records.append({
                "meeting_id": meeting_id,
                "split": split,
                "transcript": transcript_text,
                "num_utterances": len(utterance_df),
                "num_speakers": utterance_df["speaker"].nunique()
            })

    return pd.DataFrame(records)

In [6]:
def load_summaries(summary_dir):

    summaries = {}

    for filepath in glob.glob(os.path.join(summary_dir, "*.txt")):
        meeting_id = os.path.basename(filepath).replace(".txt", "")

        with open(filepath, "r", encoding="utf-8") as f:
            summaries[meeting_id] = f.read()

    return summaries

In [7]:
def load_audio_paths(audio_dir):

    audio_records = []

    for meeting in os.listdir(audio_dir):

        meeting_audio_dir = os.path.join(audio_dir, meeting, "audio")

        if not os.path.isdir(meeting_audio_dir):
            continue

        headset = None
        lapel = None

        for file in os.listdir(meeting_audio_dir):

            if "Headset" in file:
                headset = os.path.join(meeting_audio_dir, file)

            if "Lapel" in file:
                lapel = os.path.join(meeting_audio_dir, file)

        audio_records.append({
            "meeting_id": meeting,
            "audio_headset": headset,
            "audio_lapel": lapel
        })

    return pd.DataFrame(audio_records)

In [8]:
def build_dataset():
    transcripts_df = load_transcripts(TRANSCRIPT_DIR)

    summaries = load_summaries(SUMMARY_DIR)

    audio_df = load_audio_paths(AUDIO_DIR)

    # attach summaries
    transcripts_df["summary"] = transcripts_df["meeting_id"].map(summaries)

    # merge audio paths
    df = transcripts_df.merge(audio_df, on="meeting_id", how="left")

    return df

In [12]:
df = build_dataset()

print(df.shape)
print(df.head())

(137, 8)
  meeting_id  split                                         transcript  \
0    ES2002a  train  Okay . Right . Um well this is the kick-off me...   
1    ES2002b  train  Is that alright now ? Okay . Sorry ? Okay , ev...   
2    ES2002c  train  'S to do now is to decide how to fulfil what y...   
3    ES2002d  train  Okay we all all set ? Right . Well this is the...   
4    ES2005a  train  Uh , making a profit of fifty million Euros . ...   

   num_utterances  num_speakers  \
0             332             4   
1             691             4   
2             634             4   
3             854             4   
4              93             4   

                                             summary  \
0  The project manager introduced the upcoming pr...   
1  The project manager briefed the team on some n...   
2  The project manager recapped the decisions mad...   
3  The project manager recapped the decisions mad...   
4  The group discussed their initial ideas about ...   

In [10]:
print("Split counts:")
print(df["split"].value_counts())

print("\nMissing summaries:")
print(df["summary"].isna().sum())

print("\nMissing audio:")
print(df["audio_headset"].isna().sum())

Split counts:
split
train    97
valid    20
test     20
Name: count, dtype: int64

Missing summaries:
0

Missing audio:
0
